##### NOTEBOOK 2 — CAPA SILVER
##### TFM: Integración QML en pipeline DataOps
##### Juan Albornoz Carrasco — Universidad Europea de Valencia
##### Lee desde Bronze Delta Lake — Escribe Silver Delta Lake + Parquet


##### CELDA 1 — Instalación de dependencias


In [0]:
#%pip install --force-reinstall numpy==1.23.5 pyreadstat dataframe-expectations 

##### CELDA 2 — Reinicio del kernel



In [0]:
dbutils.library.restartPython()

##### CELDA 3 — Imports y configuración


In [0]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import seaborn as sns
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from delta.tables import DeltaTable

In [0]:
spark = SparkSession.builder.getOrCreate()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Rutas
bronze_path  = "/Volumes/workspace/default/nhanes/bronze"
silver_path  = "/Volumes/workspace/default/nhanes/silver_delta"
output_dir   = "/Volumes/workspace/default/nhanes/silver"

os.makedirs(output_dir, exist_ok=True)
print("Imports OK")
print(f"Bronze path:  {bronze_path}")
print(f"Silver Delta: {silver_path}")
print(f"Silver Parquet: {output_dir}")

Imports OK
Bronze path:  /Volumes/workspace/default/nhanes/bronze
Silver Delta: /Volumes/workspace/default/nhanes/silver_delta
Silver Parquet: /Volumes/workspace/default/nhanes/silver


##### CELDA 4 — Lectura desde Bronze Delta Lake
##### Silver transforma desde Bronze, no desde origen


In [0]:
df_bronze = spark.read.format("delta").load(bronze_path).toPandas()
print(f"Bronze cargado desde Delta Lake: {df_bronze.shape[0]} filas, {df_bronze.shape[1]} columnas")
print(f"Ciclos: {df_bronze['CICLO'].value_counts().to_dict()}")

Bronze cargado desde Delta Lake: 29400 filas, 162 columnas
Ciclos: {'2013-2014': 10175, '2015-2016': 9971, '2017-2018': 9254}


##### CELDA 5 — Filtros de calidad del dato
##### Filtro 1: Edad >= 18 años


In [0]:
antes = len(df_bronze)
df_silver = df_bronze[df_bronze["RIDAGEYR"] >= 18].copy()
despues = len(df_silver)
print(f"Filtro edad >= 18: {antes} → {despues} registros ({antes - despues} eliminados)")

Filtro edad >= 18: 29400 → 17961 registros (11439 eliminados)


##### CELDA 6 — Filtro 2: Subgrupo ayuno
##### LBXGLU no nulo como proxy de ayuno
##### PHAFSTMN no disponible consistentemente en los 3 ciclos


In [0]:
antes = len(df_silver)
df_silver = df_silver.dropna(subset=["LBXGLU"]).copy()
despues = len(df_silver)
print(f"Filtro ayuno (LBXGLU no nulo): {antes} → {despues} registros ({antes-despues} eliminados)")
print(f"\nEstadísticas LBXGLU post-filtro:")
print(df_silver["LBXGLU"].describe().round(2))

Filtro ayuno (LBXGLU no nulo): 17961 → 7835 registros (10126 eliminados)

Estadísticas LBXGLU post-filtro:
count    7835.00
mean      111.00
std        37.44
min        21.00
25%        94.00
50%       101.00
75%       112.00
max       479.00
Name: LBXGLU, dtype: float64


##### CELDA 7 — Filtro 3: Variable objetivo DIQ010
##### Eliminar valores 7 (No sabe) y 9 (Se rehusa) y NaN


In [0]:
antes = len(df_silver)
df_silver = df_silver[~df_silver["DIQ010"].isin([7.0, 9.0])].copy()
df_silver = df_silver.dropna(subset=["DIQ010"]).copy()
despues = len(df_silver)
print(f"Filtro DIQ010: {antes} → {despues} registros ({antes - despues} eliminados)")
print(f"\nDistribución DIQ010 post-filtro:")
print(df_silver["DIQ010"].value_counts())

Filtro DIQ010: 7831 → 7831 registros (0 eliminados)

Distribución DIQ010 post-filtro:
DIQ010
2.0    6510
1.0    1099
3.0     222
Name: count, dtype: int64


##### CELDA 8 — Binarización del TARGET
##### 1.0 → diabetes = 1 | 2.0 y 3.0 → no diabetes = 0


In [0]:
df_silver["TARGET"] = np.where(df_silver["DIQ010"] == 1.0, 1, 0)

print(f"Distribución TARGET:")
print(df_silver["TARGET"].value_counts())
print(f"\nDistribución TARGET (%):")
print(df_silver["TARGET"].value_counts(normalize=True).mul(100).round(2))

Distribución TARGET:
TARGET
0    6732
1    1099
Name: count, dtype: int64

Distribución TARGET (%):
TARGET
0    85.97
1    14.03
Name: proportion, dtype: float64


##### CELDA 9 — Exclusión de variables DIQ por leakage conceptual
##### Se excluyen antes de winsorización para no procesar variables que no formarán parte del dataset final.

In [0]:
vars_leakage = [
    "DIQ040", "DIQ050", "DIQ070",
    "DIQ160", "DIQ170", "DIQ172", "DIQ180"
]

vars_leakage_presentes = [v for v in vars_leakage if v in df_silver.columns]
df_silver = df_silver.drop(columns=vars_leakage_presentes)

print(f"Variables DIQ excluidas por leakage: {vars_leakage_presentes}")
print(f"Dataset tras exclusión leakage: {df_silver.shape}")

Variables DIQ excluidas por leakage: ['DIQ050', 'DIQ070', 'DIQ160', 'DIQ170', 'DIQ172', 'DIQ180']
Dataset tras exclusión leakage: (7831, 157)


##### CELDA 10 — Eliminar columnas con >80% missing


In [0]:
threshold = 0.80
missing_pct = df_silver.isnull().mean()
cols_eliminar = missing_pct[missing_pct > threshold].index.tolist()

print(f"Columnas con >80% missing eliminadas: {len(cols_eliminar)}")
df_silver = df_silver.drop(columns=cols_eliminar)
print(f"Dataset tras eliminar columnas sparse: {df_silver.shape}")

Columnas con >80% missing eliminadas: 66
Dataset tras eliminar columnas sparse: (7831, 91)


##### CELDA 11 — Tratamiento de outliers (winsorización IQR x3)


In [0]:
vars_numericas = df_silver.select_dtypes(include=[np.number]).columns.tolist()
vars_excluir_winsor = ["SEQN", "TARGET", "DIQ010", "RIDAGEYR", "RIAGENDR"]
vars_winsorizar = [v for v in vars_numericas if v not in vars_excluir_winsor]

outliers_corregidos = {}
for col in vars_winsorizar:
    Q1 = df_silver[col].quantile(0.25)
    Q3 = df_silver[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    n_outliers = ((df_silver[col] < lower) | (df_silver[col] > upper)).sum()
    if n_outliers > 0:
        df_silver[col] = df_silver[col].clip(lower=lower, upper=upper)
        outliers_corregidos[col] = n_outliers

print(f"Variables con outliers corregidos: {len(outliers_corregidos)}")
for col, n in sorted(outliers_corregidos.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {col}: {n} outliers winsorizados")

Variables con outliers corregidos: 67
  PAQ635: 1929 outliers winsorizados
  PAQ650: 1847 outliers winsorizados
  PAQ605: 1648 outliers winsorizados
  DMDHHSZA: 1535 outliers winsorizados
  DMDCITZN: 1150 outliers winsorizados
  SIALANG: 897 outliers winsorizados
  AIALANGA: 825 outliers winsorizados
  DMQMILIZ: 740 outliers winsorizados
  FIALANG: 688 outliers winsorizados
  MIALANG: 677 outliers winsorizados


##### CELDA 12 — Imputación de missing values
##### LightGBM: sin imputar (gestiona nativamente)
##### SVM y QSVM: mediana (continuas) + moda (categoricas)


In [0]:
vars_categoricas = ["RIAGENDR", "RIDRETH1", "RIDRETH3", "DMDEDUC2", "DMDMARTL"]
vars_categoricas_presentes = [v for v in vars_categoricas if v in df_silver.columns]

df_silver_lgbm = df_silver.copy()

df_silver_svm = df_silver.copy()

vars_continuas = [v for v in vars_numericas
                  if v not in vars_excluir_winsor
                  and v not in vars_categoricas_presentes]

for col in vars_continuas:
    if col in df_silver_svm.columns:
        mediana = df_silver_svm[col].median()
        df_silver_svm[col] = df_silver_svm[col].fillna(mediana)

for col in vars_categoricas_presentes:
    moda = df_silver_svm[col].mode()[0]
    df_silver_svm[col] = df_silver_svm[col].fillna(moda)

print(f"Silver LightGBM: {df_silver_lgbm.shape} — missing: {df_silver_lgbm.isnull().sum().sum()}")
print(f"Silver SVM/QSVM: {df_silver_svm.shape} — missing: {df_silver_svm.isnull().sum().sum()}")

Silver LightGBM: (7831, 91) — missing: 75855
Silver SVM/QSVM: (7831, 91) — missing: 0


##### CELDA 13 — Resumen final Silver


In [0]:
print("=== RESUMEN CAPA SILVER ===")
print(f"Registros finales:     {len(df_silver_svm)}")
print(f"Columnas finales:      {df_silver_svm.shape[1]}")
print(f"Casos diabetes (1):    {df_silver_svm['TARGET'].sum()} ({df_silver_svm['TARGET'].mean()*100:.1f}%)")
print(f"Casos no diabetes (0): {(df_silver_svm['TARGET']==0).sum()} ({(df_silver_svm['TARGET']==0).mean()*100:.1f}%)")
print(f"Distribucion por ciclo:")
print(df_silver_svm["CICLO"].value_counts())

missing_post = df_silver_svm.isnull().sum()
missing_post = missing_post[missing_post > 0]
if len(missing_post) > 0:
    print(f"\nColumnas con missing restante en SVM dataset:")
    print(missing_post)
else:
    print("\nSin missing values en dataset SVM/QSVM OK")

=== RESUMEN CAPA SILVER ===
Registros finales:     7831
Columnas finales:      91
Casos diabetes (1):    1099 (14.0%)
Casos no diabetes (0): 6732 (86.0%)
Distribucion por ciclo:
CICLO
2013-2014    2725
2015-2016    2571
2017-2018    2535
Name: count, dtype: int64

Sin missing values en dataset SVM/QSVM OK


##### CELDA 14 — Guardar Silver como Parquet
##### Compatibilidad con notebooks de modelado (Gold, LightGBM, SVM)


In [0]:
def normalize_and_save(df, path):
    df = df.copy()

    for col in df.columns:
        if pd.api.types.is_extension_array_dtype(df[col]):
            try:
                df[col] = df[col].astype(df[col].dtype.numpy_dtype)
            except AttributeError:
                df[col] = df[col].astype(object)

        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors='coerce')

    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, path)

normalize_and_save(df_silver_lgbm, f"{output_dir}/silver_lgbm.parquet")
normalize_and_save(df_silver_svm,  f"{output_dir}/silver_svm.parquet")

print(f"Silver LightGBM guardado: {output_dir}/silver_lgbm.parquet")
print(f"Silver SVM/QSVM guardado: {output_dir}/silver_svm.parquet")

Silver LightGBM guardado: /Volumes/workspace/default/nhanes/silver/silver_lgbm.parquet
Silver SVM/QSVM guardado: /Volumes/workspace/default/nhanes/silver/silver_svm.parquet


##### CELDA 15 — Materializar Silver en Delta Lake
##### Garantiza ACID, time travel y trazabilidad DataOps
##### Dataset SVM usado como Silver canonico (imputacion completa)


In [0]:
def normalize_dtypes(df):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_extension_array_dtype(df[col]):
            try:
                df[col] = df[col].astype(df[col].dtype.numpy_dtype)
            except AttributeError:
                df[col] = df[col].astype(object)
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

df_silver_clean = normalize_dtypes(df_silver_svm)
df_silver_spark = spark.createDataFrame(df_silver_clean)

df_silver_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

print(f"Silver materializado en Delta Lake")
print(f"Registros: {df_silver_spark.count()}")
print(f"Columnas:  {len(df_silver_spark.columns)}")
print(f"Ruta:      {silver_path}")

Silver materializado en Delta Lake
Registros: 7831
Columnas:  91
Ruta:      /Volumes/workspace/default/nhanes/silver_delta


##### CELDA 16 — Verificacion Delta Lake Silver


In [0]:
delta_table = DeltaTable.forPath(spark, silver_path)
historial = delta_table.history().toPandas()
cols_principales = ["version", "timestamp", "operation", "operationMetrics", "userName"]
display(historial[cols_principales])

version,timestamp,operation,operationMetrics,userName
6,2026-06-20T21:43:30.000Z,WRITE,"List(0, 8, 988870, 7831, 989071, 8)",juan.adrian.albornoz@gmail.com
5,2026-06-13T21:39:24.000Z,WRITE,"List(0, 8, 989071, 7831, 1005835, 8)",juan.adrian.albornoz@gmail.com
4,2026-06-13T21:25:40.000Z,WRITE,"List(0, 8, 1005835, 7831, 1004003, 8)",juan.adrian.albornoz@gmail.com
3,2026-06-01T19:46:19.000Z,WRITE,"List(0, 8, 1004003, 7831, 1004003, 8)",juan.adrian.albornoz@gmail.com
2,2026-05-31T17:37:04.000Z,WRITE,"List(0, 8, 1004003, 7831, 1004003, 8)",juan.adrian.albornoz@gmail.com
1,2026-05-31T17:35:51.000Z,WRITE,"List(0, 8, 1004003, 7831, 1003983, 8)",juan.adrian.albornoz@gmail.com
0,2026-05-22T21:06:54.000Z,WRITE,"List(0, 8, 1003983, 7831, 0, 0)",juan.adrian.albornoz@gmail.com


##### CELDA 17 — Validacion final Silver


In [0]:
print("=== VALIDACION FINAL CAPA SILVER ===")

print("\n1. Parquet:")
import os
for f in ["silver_lgbm.parquet", "silver_svm.parquet"]:
    path = f"{output_dir}/{f}"
    df_check = pd.read_parquet(path)
    print(f"   {f}: {df_check.shape}")

print("\n2. Delta Lake:")
df_verify = spark.read.format("delta").load(silver_path)
print(f"   Registros: {df_verify.count()}")
print(f"   Columnas:  {len(df_verify.columns)}")

print("\n3. Distribucion clases Delta:")
df_verify.groupBy("TARGET").count().show()

print("\n=== SILVER VALIDADO OK ===")

=== VALIDACION FINAL CAPA SILVER ===

1. Parquet:
   silver_lgbm.parquet: (7831, 91)
   silver_svm.parquet: (7831, 91)

2. Delta Lake:
   Registros: 7831
   Columnas:  91

3. Distribucion clases Delta:
+------+-----+
|TARGET|count|
+------+-----+
|     0| 6732|
|     1| 1099|
+------+-----+


=== SILVER VALIDADO OK ===


###### CELDA 18 — Verificación de leakage Silver y Gold
###### Confirma que las variables DIQ excluidas no llegaron a los datasets de modelado


In [0]:
gold_dir   = "/Volumes/workspace/default/nhanes/gold"
silver_dir = "/Volumes/workspace/default/nhanes/silver"

vars_leakage = ["DIQ040", "DIQ050", "DIQ070",
                "DIQ160", "DIQ170", "DIQ172", "DIQ180"]

print("=== VERIFICACION LEAKAGE ===\n")

# Silver
print("--- SILVER ---")
for f in ["silver_lgbm.parquet", "silver_svm.parquet"]:
    df = pd.read_parquet(f"{silver_dir}/{f}")
    presentes = [v for v in vars_leakage if v in df.columns]
    print(f"{f}: {presentes if presentes else 'LIMPIO ✅'}")

print()

# Gold
print("--- GOLD ---")
gold_files = [
    "X_train_qsvm.parquet",
    "X_train_qsvm_8features.parquet",
    "X_test_qsvm_8features.parquet",
    "y_train_qsvm.parquet",
    "y_test_qsvm.parquet",
    "X_train_lgbm.parquet",
    "X_test_lgbm.parquet",
    "y_train_lgbm.parquet",
    "y_test_lgbm.parquet",
    "X_train_svm.parquet",
    "X_test_svm.parquet",
    "y_train_svm.parquet",
    "y_test_svm.parquet",
]

for f in gold_files:
    try:
        df = pd.read_parquet(f"{gold_dir}/{f}")
        presentes = [v for v in vars_leakage if v in df.columns]
        print(f"{f}: {presentes if presentes else 'LIMPIO ✅'}")
    except:
        print(f"{f}: no encontrado")

=== VERIFICACION LEAKAGE ===

--- SILVER ---
silver_lgbm.parquet: LIMPIO ✅
silver_svm.parquet: LIMPIO ✅

--- GOLD ---
X_train_qsvm.parquet: LIMPIO ✅
X_train_qsvm_8features.parquet: LIMPIO ✅
X_test_qsvm_8features.parquet: LIMPIO ✅
y_train_qsvm.parquet: LIMPIO ✅
y_test_qsvm.parquet: LIMPIO ✅
X_train_lgbm.parquet: LIMPIO ✅
X_test_lgbm.parquet: LIMPIO ✅
y_train_lgbm.parquet: LIMPIO ✅
y_test_lgbm.parquet: LIMPIO ✅
X_train_svm.parquet: LIMPIO ✅
X_test_svm.parquet: LIMPIO ✅
y_train_svm.parquet: LIMPIO ✅
y_test_svm.parquet: LIMPIO ✅
